In [ ]:
!pip install -q neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 8.4 MB/s eta 0:00:00


In [ ]:
import neo4j

print("Neo4j Python Driver Version:", neo4j.__version__)

Neo4j Python Driver Version: 6.3.1


In [ ]:
# 1.เริ่มติดต่อใช้งาน Neo4j online server

from getpass import getpass
from neo4j import GraphDatabase

URI = "neo4j+s://af29e49d.databases.neo4j.io"
USERNAME = "af29e49d"
PASSWORD = getpass("Password: ")

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()

print("Connected successfully")

Password: ··········
Connected successfully


In [ ]:
# หา home database

result = driver.execute_query(
    "SHOW HOME DATABASE"
)

for record in result.records:
    print(record.data())

{'name': 'af29e49d', 'type': 'standard', 'aliases': [], 'access': 'read-write', 'address': 'p-mt-106bc5f59e02-10-0149.production-orch-0066.neo4j.io:7687', 'role': 'primary', 'writer': True, 'requestedStatus': 'online', 'currentStatus': 'online', 'statusMessage': '', 'constituents': []}


In [ ]:
# เก็บชื่อ database

DATABASE = result.records[0]["name"]

print("Database =", DATABASE)

Database = af29e49d


In [ ]:
# Query ทุกคำสั่งใช้

result = driver.execute_query(
    """
    RETURN 'Hello Neo4j, I am ready' AS message
    """,
    database_=DATABASE
)

print(result.records[0]["message"])

Hello Neo4j, I am ready


In [ ]:
# ทดสอบใช้งานฐานข้อมูลจริง

result = driver.execute_query(
    """
    MATCH (n)
    RETURN count(n) AS total_nodes
    """,
    database_=DATABASE
)

print("จำนวน Node =", result.records[0]["total_nodes"])

จำนวน Node = 0


In [ ]:
result = driver.execute_query(
    """
    CREATE (s:Student {
        student_id: 'S001',
        name: 'Alice'
    })

    RETURN s
    """,
    database_=DATABASE
)

print(result.records[0]["s"])

<Node element_id='4:f2ca6e80-5536-44d2-aff3-846b60c9060b:0' labels=frozenset({'Student'}) properties={'name': 'Alice', 'student_id': 'S001'}>


In [ ]:
# ตรวจสอบผลลัพธ์

result = driver.execute_query(
    """
    MATCH (s:Student)

    RETURN
        s.student_id AS id,
        s.name AS name
    """,
    database_=DATABASE
)

for record in result.records:
    print(record.data())

{'id': 'S001', 'name': 'Alice'}


In [ ]:
# ทดลอง where

result = driver.execute_query(
    """
    MATCH (s:Student)

    WHERE s.student_id = 'S001'

    RETURN
        s.name AS name
    """,
    database_=DATABASE
)

for record in result.records:
    print(record.data())

{'name': 'Alice'}


In [ ]:
# สร้าง constrain

driver.execute_query(
    """
    CREATE CONSTRAINT student_id_unique
    IF NOT EXISTS

    FOR (s:Student)

    REQUIRE s.student_id IS UNIQUE
    """,
    database_=DATABASE
)

print("Student constraint created")

Student constraint created


In [ ]:
driver.execute_query(
    """
    CREATE CONSTRAINT gamegenre_id_unique
    IF NOT EXISTS

    FOR (g:gamegenre)

    REQUIRE g.gamegenre_id IS UNIQUE
    """,
    database_=DATABASE
)

print("gamegenre constraint created")

gamegenre constraint created


In [ ]:
# เพิ่ม student หลายๆคน

students = [
    {"student_id": "S002", "name": "Bob"},
    {"student_id": "S003", "name": "Charlie"},
    {"student_id": "S004", "name": "David"},
    {"student_id": "S005", "name": "Eve"},
    {"student_id": "S006", "name": "Frank"},
    {"student_id": "S007", "name": "Grace"},
    {"student_id": "S008", "name": "Heidi"},
    {"student_id": "S009", "name": "Ivan"},
    {"student_id": "S010", "name": "Judy"}
]

In [ ]:
result = driver.execute_query(
    """
    UNWIND $students AS row

    MERGE (s:Student {
        student_id: row.student_id
    })

    SET
        s.name = row.name,

    RETURN count(s) AS total
    """,

    students=students,
    database_=DATABASE
)

print(
    "จำนวน Student ที่ประมวลผล =",
    result.records[0]["total"]
)

จำนวน Student ที่ประมวลผล = 9


In [ ]:
gamegenres = [
    {
        "gamegenre_id": "G101",
        "name": "Action"
    },
    {
        "gamegenre_id": "G102",
        "name": "RPG"
    },
    {
        "gamegenre_id": "G103",
        "name": "FPS"
    },
    {
        "gamegenre_id": "G104",
        "name": "MOBA"
    },
    {
        "gamegenre_id": "G105",
        "name": "Strategy"
    },
    {
        "gamegenre_id": "G106",
        "name": "Simulation"
    },
    {
        "gamegenre_id": "G107",
        "name": "Puzzle"
    },
    {
        "gamegenre_id": "G108",
        "name": "Racing"
    },
    {
        "gamegenre_id": "G109",
        "name": "Sports"
    },
    {
        "gamegenre_id": "G110",
        "name": "Horror"
    }
]

In [ ]:
result = driver.execute_query(
    """
    UNWIND $gamegenres AS row

    MERGE (g:gamegenre {
        gamegenre_id: row.gamegenre_id
    })

    SET
        g.name = row.name,

    RETURN count(s) AS total
    """,

    gamegenres=gamegenres,
    database_=DATABASE
)

print(
    "จำนวน gamegenre ที่ประมวลผล =",
    result.records[0]["total"]
)

จำนวน gamegenre ที่ประมวลผล = 10


In [ ]:
# ดู student และ gamegenre ทั้งหมด

result = driver.execute_query(
    """
    MATCH (n)

    WHERE n:Student OR n:gamegenre

    RETURN
        labels(n) AS type,
        properties(n) AS data
    """,
    database_=DATABASE
)

for record in result.records:
    print(record.data())

{'type': ['Student'], 'data': {'name': 'Alice', 'student_id': 'S001'}}
{'type': ['Student'], 'data': {'name': 'Bob', 'student_id': 'S002'}}
{'type': ['Student'], 'data': {'name': 'Charlie', 'student_id': 'S003'}}
{'type': ['Student'], 'data': {'name': 'David', 'student_id': 'S004'}}
{'type': ['Student'], 'data': {'name': 'Eve', 'student_id': 'S005'}}
{'type': ['Student'], 'data': {'name': 'Frank', 'student_id': 'S006'}}
{'type': ['Student'], 'data': {'name': 'Grace', 'student_id': 'S007'}}
{'type': ['Student'], 'data': {'name': 'Heidi', 'student_id': 'S008'}}
{'type': ['Student'], 'data': {'name': 'Ivan', 'student_id': 'S009'}}
{'type': ['Student'], 'data': {'name': 'Judy', 'student_id': 'S010'}}
{'type': ['gamegenre'], 'data': {'gamegenre_id': 'G101', 'name': 'Action'}}
{'type': ['gamegenre'], 'data': {'gamegenre_id': 'G102', 'name': 'RPG'}}
{'type': ['gamegenre'], 'data': {'gamegenre_id': 'G103', 'name': 'FPS'}}
{'type': ['gamegenre'], 'data': {'gamegenre_id': 'G104', 'name': 'MOBA'}

In [ ]:
# สร้าง Relationship FRIEND_OF

friendships = [
    {"student1": "S001", "student2": "S002"},
    {"student1": "S001", "student2": "S005"},
    {"student1": "S001", "student2": "S010"},
    {"student1": "S002", "student2": "S003"},
    {"student1": "S003", "student2": "S004"},
    {"student1": "S003", "student2": "S008"},
    {"student1": "S004", "student2": "S005"},
    {"student1": "S004", "student2": "S009"},
    {"student1": "S005", "student2": "S006"},
    {"student1": "S006", "student2": "S007"},
    {"student1": "S007", "student2": "S008"},
    {"student1": "S008", "student2": "S009"},
    {"student1": "S009", "student2": "S010"},
    {"student1": "S010", "student2": "S002"}
]

In [ ]:
driver.execute_query(
    """
    UNWIND $friendships AS row

    MATCH
        (a:Student {student_id: row.student1}),
        (b:Student {student_id: row.student2})

    MERGE
        (a)-[:FRIEND_OF]->(b)
    """,

    friendships=friendships,
    database_=DATABASE
)

print("สร้าง FRIEND_OF สำเร็จ")

สร้าง FRIEND_OF สำเร็จ


In [ ]:
# Query ความสัมพันธ์

result = driver.execute_query(
    """
    MATCH
        (s:Student {student_id:'S002'})
        -[:FRIEND_OF]->
        (friend:Student)

    RETURN friend.name AS friend
    """,
    database_=DATABASE
)

for record in result.records:
    print(record["friend"])

Charlie


In [ ]:
# สร้าง genres_likes Relationship

genres_likes = [
    # S001
    {"student_id": "S001", "gamegenre_id": "G101", "date": "2018-05-12"},
    {"student_id": "S001", "gamegenre_id": "G102", "date": "2020-11-20"},
    {"student_id": "S001", "gamegenre_id": "G103", "date": "2022-01-15"},

    # S002
    {"student_id": "S002", "gamegenre_id": "G103", "date": "2015-03-15"},
    {"student_id": "S002", "gamegenre_id": "G104", "date": "2017-06-18"},
    {"student_id": "S002", "gamegenre_id": "G105", "date": "2019-08-01"},

    # S003
    {"student_id": "S003", "gamegenre_id": "G102", "date": "2017-01-10"},
    {"student_id": "S003", "gamegenre_id": "G104", "date": "2021-04-25"},

    # S004
    {"student_id": "S004", "gamegenre_id": "G101", "date": "2014-02-14"},
    {"student_id": "S004", "gamegenre_id": "G108", "date": "2016-09-18"},
    {"student_id": "S004", "gamegenre_id": "G109", "date": "2018-12-05"},

    # S005
    {"student_id": "S005", "gamegenre_id": "G106", "date": "2019-03-30"},
    {"student_id": "S005", "gamegenre_id": "G110", "date": "2022-10-31"},

    # S006
    {"student_id": "S006", "gamegenre_id": "G105", "date": "2013-11-11"},
    {"student_id": "S006", "gamegenre_id": "G107", "date": "2016-07-20"},

    # S007
    {"student_id": "S007", "gamegenre_id": "G102", "date": "2018-04-05"},
    {"student_id": "S007", "gamegenre_id": "G104", "date": "2020-08-14"},
    {"student_id": "S007", "gamegenre_id": "G106", "date": "2023-02-28"},

    # S008
    {"student_id": "S008", "gamegenre_id": "G103", "date": "2016-10-10"},
    {"student_id": "S008", "gamegenre_id": "G108", "date": "2019-05-22"},

    # S009
    {"student_id": "S009", "gamegenre_id": "G101", "date": "2015-08-08"},
    {"student_id": "S009", "gamegenre_id": "G107", "date": "2018-12-12"},
    {"student_id": "S009", "gamegenre_id": "G110", "date": "2021-09-01"},

    # S010
    {"student_id": "S010", "gamegenre_id": "G105", "date": "2017-04-01"},
    {"student_id": "S010", "gamegenre_id": "G109", "date": "2020-10-15"}
]

In [ ]:
# สร้าง Relationship

driver.execute_query(
    """
    UNWIND $genres_likes AS row

    MATCH
        (s:Student {student_id: row.student_id}),
        (g:gamegenre {gamegenre_id: row.gamegenre_id})

    MERGE
        (s)-[r:GENRES_LIKES]->(g)

    SET
        r.genres_like_date = date(row.date)
    """,

    genres_likes=genres_likes,
    database_=DATABASE
)

print("สร้าง GENRES_LIKES สำเร็จ")

สร้าง GENRES_LIKES สำเร็จ


In [ ]:
# ดูว่าใครยืมอะไร

result = driver.execute_query(
    """
    MATCH
        (s:Student)
        -[r:GENRES_LIKES]->
        (g:gamegenre)

    RETURN
        s.name AS student,
        g.name AS gamegenre,
        r.genres_like_date AS date

    ORDER BY student, date
    """,
    database_=DATABASE
)

for record in result.records:
    print(
        record["student"],
        "->",
        record["gamegenre"],
        "->",
        record["date"]
    )

Alice -> Action -> 2018-05-12
Alice -> RPG -> 2020-11-20
Alice -> FPS -> 2022-01-15
Bob -> FPS -> 2015-03-15
Bob -> MOBA -> 2017-06-18
Bob -> Strategy -> 2019-08-01
Charlie -> RPG -> 2017-01-10
Charlie -> MOBA -> 2021-04-25
David -> Action -> 2014-02-14
David -> Racing -> 2016-09-18
David -> Sports -> 2018-12-05
Eve -> Simulation -> 2019-03-30
Eve -> Horror -> 2022-10-31
Frank -> Strategy -> 2013-11-11
Frank -> Puzzle -> 2016-07-20
Grace -> RPG -> 2018-04-05
Grace -> MOBA -> 2020-08-14
Grace -> Simulation -> 2023-02-28
Heidi -> FPS -> 2016-10-10
Heidi -> Racing -> 2019-05-22
Ivan -> Action -> 2015-08-08
Ivan -> Puzzle -> 2018-12-12
Ivan -> Horror -> 2021-09-01
Judy -> Strategy -> 2017-04-01
Judy -> Sports -> 2020-10-15


In [ ]:
# นำผลลัพธ์มาแสดงเป็นตาราง Pandas

import pandas as pd

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (s:Student)
        -[r:GENRES_LIKES]->
        (g:gamegenre)

    RETURN
        s.student_id AS student_id,
        s.name AS student,
        g.gamegenre_id AS gamegenre_id,
        g.name AS gamegenre,
        r.genres_like_date AS genres_like_date

    ORDER BY student_id
    """,
    database_=DATABASE
)

data = [
    record.data()
    for record in result.records
]

df = pd.DataFrame(data)

df

,student_id,student,gamegenre_id,gamegenre,genres_like_date
0,S001,Alice,G101,Action,2018-05-12
1,S001,Alice,G102,RPG,2020-11-20
2,S001,Alice,G103,FPS,2022-01-15
3,S002,Bob,G103,FPS,2015-03-15
4,S002,Bob,G104,MOBA,2017-06-18
5,S002,Bob,G105,Strategy,2019-08-01
6,S003,Charlie,G102,RPG,2017-01-10
7,S003,Charlie,G104,MOBA,2021-04-25
8,S004,David,G101,Action,2014-02-14
9,S004,David,G108,Racing,2016-09-18


In [ ]:
# Traversal หลายทอด

result = driver.execute_query(
    """
    MATCH
        (me:Student {student_id:'S001'})
        -[:FRIEND_OF]-
        (friend:Student)
        -[:GENRES_LIKES]->
        (genre:gamegenre)

    RETURN
        me.name AS me,
        friend.name AS friend,
        genre.name AS genre

    ORDER BY friend, genre
    """,
    database_=DATABASE
)

for record in result.records:
    print(
        record["me"],
        "-> เพื่อน:",
        record["friend"],
        "-> ชอบแนวเกม:",
        record["genre"]
    )

Alice -> เพื่อน: Bob -> ชอบแนวเกม: FPS
Alice -> เพื่อน: Bob -> ชอบแนวเกม: MOBA
Alice -> เพื่อน: Bob -> ชอบแนวเกม: Strategy
Alice -> เพื่อน: Eve -> ชอบแนวเกม: Horror
Alice -> เพื่อน: Eve -> ชอบแนวเกม: Simulation
Alice -> เพื่อน: Judy -> ชอบแนวเกม: Sports
Alice -> เพื่อน: Judy -> ชอบแนวเกม: Strategy


In [ ]:
# Aggregation

result = driver.execute_query(
    """
    MATCH
        (s:Student)
        -[:GENRES_LIKES]->
        (g:gamegenre)

    RETURN
        s.name AS student,
        count(g) AS total_genres

    ORDER BY total_genres DESC
    """,
    database_=DATABASE
)

for record in result.records:
    print(
        record["student"],
        "=",
        record["total_genres"],
        "แนวเกม"
    )

Alice = 3 แนวเกม
Bob = 3 แนวเกม
David = 3 แนวเกม
Grace = 3 แนวเกม
Ivan = 3 แนวเกม
Charlie = 2 แนวเกม
Eve = 2 แนวเกม
Frank = 2 แนวเกม
Heidi = 2 แนวเกม
Judy = 2 แนวเกม


In [ ]:
# หนังสือเล่มไหนได้รับความนิยมมากที่สุด

result = driver.execute_query(
    """
    MATCH
        (s:Student)
        -[:GENRES_LIKES]->
        (g:gamegenre)

    RETURN
        g.name AS gamegenre,
        count(s) AS total_students

    ORDER BY gamegenre DESC
    """,
    database_=DATABASE
)

pd.DataFrame(
    [record.data() for record in result.records]
)

,gamegenre,total_students
0,Strategy,3
1,Sports,2
2,Simulation,2
3,Racing,2
4,RPG,3
5,Puzzle,2
6,MOBA,3
7,Horror,2
8,FPS,3
9,Action,3


In [ ]:
# Recommendation System

result = driver.execute_query(
    """
    MATCH
        (me:Student {student_id:$student_id})
        -[:FRIEND_OF]-
        (friend:Student)
        -[:GENRES_LIKES]->
        (genre:gamegenre)

    WHERE NOT EXISTS {

        MATCH
            (me)-[:GENRES_LIKES]->(genre)

    }

    RETURN
        genre.gamegenre_id AS gamegenre_id,
        genre.name AS recommendation,
        count(DISTINCT friend) AS friend_score

    ORDER BY
        friend_score DESC,
        recommendation
    """,

    student_id="S001",
    database_=DATABASE
)

recommend_df = pd.DataFrame(
    [record.data() for record in result.records]
)

recommend_df

,gamegenre_id,recommendation,friend_score
0,G105,Strategy,2
1,G110,Horror,1
2,G104,MOBA,1
3,G106,Simulation,1
4,G109,Sports,1


In [ ]:
# Python Function

def recommend_genres(student_id):

    query = """
    MATCH
        (me:Student {student_id:$student_id})
        -[:FRIEND_OF]-
        (friend:Student)
        -[:GENRES_LIKES]->
        (gamegenre:gamegenre)

    WHERE NOT EXISTS {

        MATCH
            (me)-[:GENRES_LIKES]->(gamegenre)

    }

    RETURN
        gamegenre.gamegenre_id AS gamegenre_id,
        gamegenre.name AS recommendation,
        count(DISTINCT friend) AS score

    ORDER BY
        score DESC,
        recommendation
    """

    result = driver.execute_query(
        query,
        student_id=student_id,
        database_=DATABASE
    )

    return pd.DataFrame(
        [record.data() for record in result.records]
    )

In [ ]:
recommend_genres("S002")

,gamegenre_id,recommendation,score
0,G102,RPG,2
1,G101,Action,1
2,G109,Sports,1


In [ ]:
# Function ค้นหานักเรียน

def find_student(student_id):

    result = driver.execute_query(
        """
        MATCH (s:Student {student_id:$student_id})

        RETURN
            s.student_id AS id,
            s.name AS name
        """,

        student_id=student_id,
        database_=DATABASE
    )

    if len(result.records) == 0:
        return "ไม่พบนักเรียน"

    return result.records[0].data()

In [ ]:
find_student("S001")

{'id': 'S001', 'name': 'Alice'}

In [ ]:
# Function ดูแนวเกมที่นักเรียนชอบ
def game_genres_likes(student_id):

    result = driver.execute_query(
        """
        MATCH
            (s:Student {student_id:$student_id})
            -[r:GENRES_LIKES]->
            (g:gamegenre)

        RETURN
            g.gamegenre_id AS gamegenre_id,
            g.name AS name,
            r.genres_like_date AS genres_like_date

        ORDER BY genres_like_date
        """,

        student_id=student_id,
        database_=DATABASE
    )

    return pd.DataFrame(
        [record.data() for record in result.records]
    )

In [ ]:
game_genres_likes("S002")

,gamegenre_id,name,genres_like_date
0,G103,FPS,2015-03-15
1,G104,MOBA,2017-06-18
2,G105,Strategy,2019-08-01


In [ ]:
# ดู Graph ใน Neo4j Aura

result = driver.execute_query(
    """
    MATCH (n)

    OPTIONAL MATCH (n)-[r]->()

    RETURN
        count(DISTINCT n) AS nodes,
        count(r) AS relationships
    """,
    database_=DATABASE
)

print(result.records[0].data())

{'nodes': 20, 'relationships': 39}


In [ ]:
driver.close()

print("Neo4j connection closed")

Neo4j connection closed
